<a href="https://colab.research.google.com/github/hiiamlars/master_thesis/blob/main/tvd_mi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Setup

In [1]:
# @title Dependencies

# Installations of third-party libraries
!pip install openai -q
!pip install anthropic -q
!pip install together -q

# First-party imports
import os
import json
import csv
import time
import random
from typing import List, Dict, Any, Optional, Tuple

# Third-party imports
import pandas as pd
from openai import OpenAI
from anthropic import Anthropic
from together import Together
from google.colab import drive

print("\nInstalling and loading dependencies complete!")

╭───────────────────────────────────────────── 🚀 New SDK Available ──────────────────────────────────────────────╮
│ Together Python SDK 2.0 is now available!                                                                       │
│                                                                                                                 │
│ Install the beta:                                                                                               │
│ pip install --pre together  or  uv add together --prerelease allow                                              │
│                                                                                                                 │
│ New SDK: ]8;id=366232;https://github.com/togethercomputer/together-py\https://github.com/togethercomputer/together-py]8;;\                                                        │
│ Migration guide: ]8;id=343795;https://docs.together.ai/docs/pythonv2-migration-guide\https://docs.together.ai/docs/pythonv2-migration-guide]8;;\                                         │
│                                                                                                                 │
│ This package will be maintained until January 2026.                                                             │
│ Set TOGETHER_NO_BANNER=1 to hide this message.                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


Installing and loading dependencies complete!


In [2]:
# @title Paths

# Mount GDrive
drive.mount('/content/drive')

# Set working directory
WORKING_DIR = "/content/drive/MyDrive/Thesis/"

# Create working directory (if not already existing)
os.makedirs(WORKING_DIR, exist_ok=True)
print(f"Ensured working directory exists at: {WORKING_DIR}.")

# Define and create PAPER_DIR for paper content
PAPER_DIR = os.path.join(WORKING_DIR, "iclr/final")
os.makedirs(PAPER_DIR, exist_ok=True)
print(f"Ensured raw directory exists at: {PAPER_DIR}.")

# Define and create LLM_DIR for llm based reviews
LLM_DIR = os.path.join(WORKING_DIR, "llm_reviews")
os.makedirs(LLM_DIR, exist_ok=True)
print(f"Ensured llm review directory exists at: {LLM_DIR}.")

# Define and create RES_DIR for the result file
RES_DIR = os.path.join(WORKING_DIR, "tvd_mi")
os.makedirs(RES_DIR, exist_ok=True)
print(f"Ensured result directory exists at: {RES_DIR}.")

# Define and create RES_DIR for the log file
LOG_DIR = os.path.join(WORKING_DIR, "tvd_mi")
os.makedirs(LOG_DIR, exist_ok=True)
print(f"Ensured logging directory exists at: {LOG_DIR}.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Ensured working directory exists at: /content/drive/MyDrive/Thesis/.
Ensured raw directory exists at: /content/drive/MyDrive/Thesis/iclr/final.
Ensured llm review directory exists at: /content/drive/MyDrive/Thesis/llm_reviews.
Ensured result directory exists at: /content/drive/MyDrive/Thesis/tvd_mi.
Ensured logging directory exists at: /content/drive/MyDrive/Thesis/tvd_mi.


In [3]:
# @title Definitions

# Active model
MODEL = "gpt-4o-mini"
# MODEL = "claude-3-5-haiku-20241022"
# MODEL = "meta-llama/Llama-3.3-70B-Instruct-Turbo"

# Depay for retry
TIMEOUT_SECONDS = 1200

# API Keys
OPENAI_KEY = "OPENAI_KEY_PLACEHOLDER"
ANTHROPIC_KEY = "ANTHROPIC_KEY_PLACEHOLDER"
TOGETHER_KEY = "TOGETHER_KEY_PLACEHOLDER"

# Define the type of TVD-MI score to calculate using a numerical option
# 1: "paper / human review"
# 2: "paper / llm review"
# 3: "human review / llm review"
# 4: "llm review / llm review"
TVD_MI_SCORE_OPTION = 2

# Define col names for the result file
RES_COL = ['paper_id', 'response_a', 'response_b', 'prompt_1', 'raw_response_1', 'structured_response_1', 'prompt_2', 'raw_response_2', 'structured_response_2', 'tvd_mi_score_1', 'tvd_mi_score_2', 'avg_tvd_mi_score']

SIMULATION_ACTIVE = True # SIMULATION_ACTIVE = False activates the API calls

print("\nDefintions complete!")

In [4]:
# @title Load data

# Read dataset_paper
dataset_paper = pd.read_parquet(os.path.join(PAPER_DIR, "dataset_paper.parquet"))

# Select 'paper_id' and the target columns for paper content and human reviews
paper_content = dataset_paper[['paper_id', 'parsed_text', 'human_review_1', 'human_review_2', 'human_review_3']]

# Check paper_content
print(paper_content.head(3))

# Read llm reviews
dataset_llm = pd.read_csv(os.path.join(LLM_DIR, "llm_sim_results.csv"))

# Dynamically select 'paper_id' and all columns that start with 'review_text_'
dataset_llm_cols = ['paper_id'] + [col for col in dataset_llm.columns if col.startswith('llm_review_')]
llm_review_content = dataset_llm[dataset_llm_cols]

# Check dataset_llm
print(llm_review_content.head(3))

print("\nData loaded!")

      paper_id                                        parsed_text
0  vuD2xEtxZcj  INTRODUCTION Pruning Deep Neural Networks (DNN...
1  WoByU5W5te0  INTRODUCTION Recently, representing a 3D scene...
2  XZRmNjUMj0c  INTRODUCTION Neural Architecture Search (NAS) ...
   document_id                                        review_text
0  vuD2xEtxZcj  {\n  "model": "gpt-5-mini-2025-08-07",\n  "tem...
1  WoByU5W5te0  {\n  "model": "gpt-5-mini-2025-08-07",\n  "tem...
2  XZRmNjUMj0c  {\n  "model": "gpt-5-mini-2025-08-07",\n  "tem...


In [5]:
# @title Merge data

merged_df = pd.merge(paper_content, llm_reviews, left_on='paper_id', right_on='paper_id', how='outer')
print(merged_df.head(3))
print(merged_df.shape)

print("\nData merged!")

     paper_id                                        parsed_text document_id  \
0  -0tPmzgXS5  INTRODUCTION Video recognition methods has evo...  -0tPmzgXS5   
1  -0tPmzgXS5  INTRODUCTION Video recognition methods has evo...  -0tPmzgXS5   
2  -0tPmzgXS5  INTRODUCTION Video recognition methods has evo...  -0tPmzgXS5   

                                         review_text  
0  {\n  "model": "gpt-5-mini-2025-08-07",\n  "tem...  
1  {\n  "model": "gpt-5-mini-2025-08-07",\n  "tem...  
2  {\n  "model": "gpt-5-mini-2025-08-07",\n  "tem...  
(111664, 4)


# API

In [6]:
# @title Define functions

# Cache for critic scores to avoid redundant API calls
cache = {}

def generate_tvd_mi_prompt(response_a: str, response_b: str) -> str:

    """Generate prompt for TVD-MI critic"""

    prompt = f"""You are evaluating whether two responses could be from agents working on the same task.

Task description: Scientific paper peer review task

Response A: {response_a}

Response B: {response_b}

Do these responses show evidence of coming from the same task/source? Consider:
- Shared specific details, facts, or entities
- Similar topics or themes
- Overlapping information that would be unlikely if from different sources

Respond in two sentences or less. Put the information gain at the end in brackets in this format:

Rate the information gain:
- [[Significant Gain]]: Clear evidence they're from the same source (many shared specifics)
- [[Little Gain]]: Some shared elements but also differences
- [[No Gain]]: No evidence of shared source (could be from completely different tasks)""" # Should I adapt the first sentence for my own purpose?

    return prompt

def interpret_tvd_mi_response(response: str) -> float:

    """Convert LLM response to numeric score"""

    response = response.strip().lower()

    if "[significant gain]" in response:
        return 1.0
    elif "[little gain]" in response:
        return 0.25
    elif "[no gain]" in response:
        return 0.0
    elif "[TOTAL FAILURE]" in response:
        return "TOTAL FAILURE"
    else:
        # Default to no gain if response is unclear
        print(f"Warning: Unclear response '{response}', defaulting to [[No Gain]]")
        return 0.0

class LLMClient:
    """
    Unified client handling Simulation, Caching, and Routing
    for OpenAI, Anthropic, and Together.
    """

    def __init__(self, simulate: bool = True, seed: int = 7):
        self.simulate = simulate
        self.rng = random.Random(seed)
        self.cache = {}
        self._openai = None
        self._anthropic = None
        self._together = None

    def _maybe_init_clients(self):
        """
        Initialize clients if simulation is off
        """
        if self.simulate:
            return

        if self._openai is None and "OPENAI_KEY" in globals():
            self._openai = OpenAI(api_key=OPENAI_KEY)
        if self._anthropic is None and "ANTHROPIC_KEY" in globals():
            self._anthropic = Anthropic(api_key=ANTHROPIC_KEY)
        if self._together is None and "TOGETHER_KEY" in globals():
            self._together = Together(api_key=TOGETHER_KEY)

    def call(
        self,
        model: str,
        prompt: str,
        temperature: float = 0.0,
        max_tokens: int = 500,
        max_retries: int = 5
    ) -> Tuple[str, Dict[str, Any]]:
        """
        Call the API / simulate with the given parameters
        """

        self._maybe_init_clients()

        # Simulation
        if self.simulate:

            # Create artificial response
            raw = {
                "model": model,
                "temperature": temperature,
                "max_tokens": max_tokens,
                "provider": "simulated",
                "response": (
                    f"[significant gain]"
                )
            }

            # Make the response human readable
            structured = json.dumps(raw, indent=2)

            return structured, raw

        # APIs
        for attempt in range(max_retries):
            try:

                # Create result items
                response_text = ""
                provider = ""

                # ANTHROPIC
                if "claude" in model.lower():
                    resp = self._anthropic.messages.create(
                        model=model,
                        max_tokens=max_tokens,
                        temperature=temperature,
                        messages=[{"role": "user", "content": prompt}]
                    )
                    response_text = resp.content[0].text
                    provider = "anthropic"

                # OPENAI
                elif "gpt" in model.lower():
                    resp = self._openai.chat.completions.create(
                        model=model,
                        messages=[{"role": "user", "content": prompt}],
                        temperature=temperature,
                        max_tokens=max_tokens
                    )
                    response_text = resp.choices[0].message.content
                    provider = "openai"

                # TOGETHER
                else:
                    resp = self._together.chat.completions.create(
                        model=model,
                        messages=[{"role": "user", "content": prompt}],
                        temperature=temperature,
                        max_tokens=max_tokens
                    )
                    response_text = resp.choices[0].message.content
                    provider = "together"

                # Create full response
                raw = {
                    "model": model,
                    "temperature": temperature,
                    "max_tokens": max_tokens,
                    "provider": provider,
                    "response": response_text
                }

                # Make the response human readable
                structured = json.dumps(raw, indent=2)

                return structured, raw

            # Run again after delay until max_retries
            except Exception as e:
                wait_time = (2 ** attempt) + random.random()
                print(f"Error on attempt {attempt+1}: {e}. Retrying in {wait_time:.2f}s...")
                time.sleep(wait_time)

        # Create error in case of failure till max_retries
        error_raw = {"status": "error", "model": model, "response": "[TOTAL FAILURE]"}

        # Make error human readable
        return json.dumps(error_raw, indent=2), error_raw

def configure_tvd_mi_run(option: int, res_dir: str, log_dir: str) -> Tuple[List[str], str, str]:
    """
    Configures the column pair and output file names based on the TVD-MI score option.
    Randomly selects human/LLM review columns if multiple are available.
    """
    column_pair = []
    file_suffix = ""
    score_type_label = ""

    # Dynamically get all human review columns
    all_human_review_cols = [col for col in paper_content.columns if col.startswith('human_review_')]
    # Dynamically get all LLM review columns
    all_llm_review_cols = [col for col in llm_review_content.columns if col.startswith('llm_review_')]

    # Randomly select one human review column if available
    selected_human_review_col = random.choice(all_human_review_cols) if all_human_review_cols else ""
    # Randomly select one LLM review column if available
    selected_llm_review_col = random.choice(all_llm_review_cols) if all_llm_review_cols else ""

    if option == 1:
        score_type_label = "paper / human review"
        if not selected_human_review_col:
            raise ValueError("No human review columns available for 'paper / human review' comparison.")
        column_pair = ["parsed_text", selected_human_review_col]
        file_suffix = f"paper_{selected_human_review_col}"

    elif option == 2:
        score_type_label = "paper / llm review"
        if not selected_llm_review_col:
            raise ValueError("No LLM review columns available for 'paper / llm review' comparison.")
        column_pair = ["parsed_text", selected_llm_review_col]
        file_suffix = f"paper_{selected_llm_review_col}"

    elif option == 3:
        score_type_label = "human review / llm review"
        if not selected_human_review_col or not selected_llm_review_col:
            raise ValueError("Not enough human/LLM review columns available for 'human review / llm review' comparison.")
        column_pair = [selected_human_review_col, selected_llm_review_col]
        file_suffix = f"h_{selected_human_review_col}_l_{selected_llm_review_col}"

    elif option == 4:
        score_type_label = "llm review / llm review"
        # Check if there are sufficient llm reviews for this case
        if len(all_llm_review_cols) < 2:
            raise ValueError("Not enough LLM review columns available for 'llm review / llm review' comparison (need at least two distinct ones).")

        # Select the first LLM review column
        selected_llm_review_col_1 = selected_llm_review_col

        # Select a second, *different* LLM review column
        remaining_llm_review_cols = [col for col in all_llm_review_cols if col != selected_llm_review_col_1]
        selected_llm_review_col_2 = random.choice(remaining_llm_review_cols)

        column_pair = [selected_llm_review_col_1, selected_llm_review_col_2]
        file_suffix = f"llm_{selected_llm_review_col_1}_llm_{selected_llm_review_col_2}"
    else:
        raise ValueError(f"Invalid TVD_MI_OPTION: {option}. Please choose between 1 and 4.")

    output_path = os.path.join(res_dir, f"tvd_mi_scores_{file_suffix}.csv")
    log_path = os.path.join(log_dir, f"tvd_mi_log_{file_suffix}.csv")

    print(f"\nConfigured for TVD-MI type: '{score_type_label}'")
    print(f"Using columns: {column_pair[0]} and {column_pair[1]}")
    print(f"Results will be saved to: {output_path}")
    print(f"Logs will be saved to: {log_path}")

    return column_pair, output_path, log_path

print("\nFunctions defined!")

In [ ]:
# @title Run TVD-MI calculation

# Activate/Deactivate simulation/API calls
llm_client = LLMClient(simulate=SIMULATION_ACTIVE)

# Configure the run based on the TVD_MI_OPTION
column_pair, result_path, log_path = configure_tvd_mi_run(TVD_MI_SCORE_OPTION, RES_DIR, LOG_DIR)

# Select prompt input based on column_pair
response_a_col_name = column_pair[0]
response_b_col_name = column_pair[1]

# Information output
print(f"Prompt input from the columns {response_a_col_name} and {response_b_col_name}", flush = True)

# Check if result file exists to determine if header needs to be written
results_file_exists = os.path.exists(result_path)

# Check if log file exists to determine if header needs to be written
log_file_exists = os.path.exists(log_file_path)

# Information output
print(f"Processing and saving results to: {result_path}", flush = True)
print(f"Logging successful iterations to: {log_path}", flush = True)

# Open both files in append mode
with open(result_path, 'a', newline='', encoding='utf-8') as f_results, \
     open(log_path, 'a', newline='', encoding='utf-8') as f_log:

    writer_results = csv.DictWriter(f_results, fieldnames=RES_COL)
    writer_log = csv.writer(f_log)

    # Write headers only if files didn't exist
    if not results_file_exists:
        writer_results.writeheader()

    if not log_file_exists:
        writer_log.writerow(['paper_id'])

    # Get total number of rows for progress tracking
    total_rows = len(merged_df)

    # Iterate through each row of the merged_df (currently in test mode with the first rows)
    for i, row in merged_df.head(5).iterrows():

        # Print progress message
        print(f"Processing row {i+1}/{total_rows} (paper_id: {row['paper_id']})", flush = True)

        paper_id = row['paper_id'] # ID of the paper

        # Dynamically select response_a and response_b based on COLUMN_PAIR_CONFIG
        response_a = str(row[response_a_col_name]) if pd.notna(row[response_a_col_name]) else ""
        response_b = str(row[response_b_col_name]) if pd.notna(row[response_b_col_name]) else ""

        # Generate the TVD-MI prompt for order A, B
        tvd_mi_prompt_1 = generate_tvd_mi_prompt(response_a, response_b)

        # Call the LLMClient for the first time A, B
        structured_response_1, raw_response_1 = llm_client.call(
            model=MODEL,
            prompt=tvd_mi_prompt_1,
            temperature=0.0,
            max_tokens=500
        )
        print(f"LLM client call 1 completed for paper_id: {paper_id}", flush = True)

        # Interpret the LLM's response to get the TVD-MI score A, B
        llm_critic_response_text_1 = raw_response_1.get('response')
        tvd_mi_score_1 = interpret_tvd_mi_response(llm_critic_response_text_1)

        # Generate the TVD-MI prompt for order B, A
        tvd_mi_prompt_2 = generate_tvd_mi_prompt(response_b, response_a)

        # Call the LLMClient for the second time B, A
        structured_response_2, raw_response_2 = llm_client.call(
            model=MODEL,
            prompt=tvd_mi_prompt_2,
            temperature=0.0,
            max_tokens=500
        )
        print(f"LLM client call 2 completed for paper_id: {paper_id}", flush = True)

        # Interpret the LLM's response to get the TVD-MI score B, A
        llm_critic_response_text_2 = raw_response_2.get('response')
        tvd_mi_score_2 = interpret_tvd_mi_response(llm_critic_response_text_2)

        print(f"Response conversion completed for paper_id: {paper_id}", flush = True)

        # Calculate the average TVD-MI score or set to 'TOTAL FAILURE'
        if tvd_mi_score_1 == "TOTAL FAILURE" or tvd_mi_score_2 == "TOTAL FAILURE":
            avg_tvd_mi_score = "TOTAL FAILURE"
        else:
            avg_tvd_mi_score = (tvd_mi_score_1 + tvd_mi_score_2) / 2

        # Prepare data for writing, ensuring dicts are dumped to strings for CSV
        data_to_write = {
            'paper_id': paper_id,
            'response_a': response_a,
            'response_b': response_b,
            'prompt_1': tvd_mi_prompt_1,
            'raw_response_1': json.dumps(raw_response_1),
            'structured_response_1': structured_response_1,
            'prompt_2': tvd_mi_prompt_2,
            'raw_response_2': json.dumps(raw_response_2),
            'structured_response_2': structured_response_2,
            'tvd_mi_score_1': tvd_mi_score_1,
            'tvd_mi_score_2': tvd_mi_score_2,
            'avg_tvd_mi_score': avg_tvd_mi_score
        }

        # Write / append files
        try:
            # Result file
            writer_results.writerow(data_to_write)
            # Only log if the average score is not "TOTAL FAILURE"
            if avg_tvd_mi_score != "TOTAL FAILURE":
                # Log file
                writer_log.writerow([paper_id])
            print(f"Successfully wrote results and log for paper_id: {paper_id}")
        except Exception as e:
            print(f"FAILED to write results/log for paper_id: {paper_id}. Error: {e}")

print("\nProcessing complete!")

In [8]:
# @title Check result

# result.csv-file
tvd_mi_results_df = pd.read_csv(result_path)

# Check results
display(tvd_mi_results_df.head(5))

# log.csv-file
tvd_mi_log_df = pd.read_csv(log_path)

# Check logs
display(tvd_mi_log_df.head(5))

print("\nProcessing complete!")

,paper_id,response_a,response_b,prompt,raw_response,structured_response,tvd_mi_score
0,-0tPmzgXS5,INTRODUCTION Video recognition methods has evo...,"{\n ""model"": ""gpt-5-mini-2025-08-07"",\n ""tem...",You are evaluating whether two responses could...,"{""model"": ""gpt-4o-mini"", ""temperature"": 0.0, ""...","{\n ""model"": ""gpt-4o-mini"",\n ""temperature"":...",1.0
1,-0tPmzgXS5,INTRODUCTION Video recognition methods has evo...,"{\n ""model"": ""gpt-5-mini-2025-08-07"",\n ""tem...",You are evaluating whether two responses could...,"{""model"": ""gpt-4o-mini"", ""temperature"": 0.0, ""...","{\n ""model"": ""gpt-4o-mini"",\n ""temperature"":...",1.0
2,-0tPmzgXS5,INTRODUCTION Video recognition methods has evo...,"{\n ""model"": ""gpt-5-mini-2025-08-07"",\n ""tem...",You are evaluating whether two responses could...,"{""model"": ""gpt-4o-mini"", ""temperature"": 0.0, ""...","{\n ""model"": ""gpt-4o-mini"",\n ""temperature"":...",1.0
3,-0tPmzgXS5,INTRODUCTION Video recognition methods has evo...,"{\n ""model"": ""gpt-5-mini-2025-08-07"",\n ""tem...",You are evaluating whether two responses could...,"{""model"": ""gpt-4o-mini"", ""temperature"": 0.0, ""...","{\n ""model"": ""gpt-4o-mini"",\n ""temperature"":...",1.0
4,-0tPmzgXS5,INTRODUCTION Video recognition methods has evo...,"{\n ""model"": ""gpt-5-nano-2025-08-07"",\n ""tem...",You are evaluating whether two responses could...,"{""model"": ""gpt-4o-mini"", ""temperature"": 0.0, ""...","{\n ""model"": ""gpt-4o-mini"",\n ""temperature"":...",1.0


,paper_id
0,-0tPmzgXS5
1,-0tPmzgXS5
2,-0tPmzgXS5
3,-0tPmzgXS5
4,-0tPmzgXS5



Processing complete!


# References

Robertson, Z., & Koyejo, S. (2025). Let's Measure Information Step-by-Step: LLM-Based Evaluation Beyond Vibes. *arXiv preprint arXiv:2508.05469.*

The code can be accessed [here](https://github.com/zrobertson466920/llm-peer-prediction/tree/main).